# CHB-MIT 单通道预测 + 六通道教师蒸馏（alpha 扫描）

`colab/` 文件夹放在 MyDrive 根目录，把这个笔记本上传到 Colab，挂载后按顺序跑。

- 代码：`/content/drive/MyDrive/colab`
- EDF：`/content/chbmit`（跑完一个患者删一个）
- 结果：`/content/drive/MyDrive/colab/results/{pid}/`

每个患者：LOOCV 折 → 先训六通道教师（ictal vs 间期，1:1）→ 冻结 → 对每个 alpha 训六个单电极学生 → 用验证集概率训电极融合模型 → 测试。
归一化每患者每折一套，teacher 和 student 共用，统计池排除两边的 val/test 窗。

In [ ]:
!pip install -q mne

In [ ]:
import gc
import json
import shutil
import sys
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

CODE = Path("/content/drive/MyDrive/colab")
DATA = Path("/content/chbmit")
OUT = CODE / "results_distill"

sys.path.insert(0, str(CODE))
DATA.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import tools as t
import colab_tools as ct
from models import ConvBILSTMClassify, Detection

print("device =", t.DEVICE)

In [ ]:
ELECS = ["Fp1-F7", "Fp2-F8", "T7-P7", "T8-P8", "P7-O1", "P8-O2"]
PATIENTS = ["chb%02d" % i for i in range(1, 25)]
# PATIENTS = ["chb01"]                # 调试时只跑一个

SPH, SOP = 2 * 60, 30 * 60           # 发作前留白 2 min，发作前期 30 min
INTER_THR = 2 * 60 * 60              # 发作间期离发作至少 2 h
WIN_LEN, WIN_OL = 4, 2               # 4 s 窗口，50% 重叠
ENC_VAL, CLS_VAL = 0.1, 0.1
SEED = 0

ALPHAS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]   # FGL: alpha*CE + (1-alpha)*KL，1.0 是无蒸馏对照
T = 2

# teacher：按现在这版（每折标准化 + 100 epoch/patience 15 + ReduceLROnPlateau）
LR_T = 1e-3
EPOCHS_T, PATIENCE_T = 100, 15

# student：完全照 261105，保证 alpha=1 就是那次的无蒸馏基线
#   输入不标准化（原始带通信号）、dropout 0.7 + temporal attention、weight_decay 1e-3、
#   无 scheduler、30 epoch / patience 10
LR_S, WD_S = 1e-4, 1e-3
EPOCHS_S, PATIENCE_S = 30, 10
HIDDEN, DROPOUT = 20, 0.7
CHAN_ATTEN, TEMP_ATTEN = False, True

BATCH = 32

CONFIG = dict(
    alphas=ALPHAS, T=T, batch_size=BATCH,
    teacher=dict(lr=LR_T, epochs=EPOCHS_T, patience=PATIENCE_T, normalize=True,
                 scheduler="ReduceLROnPlateau(min, factor=0.5, patience=5, min_lr=1e-6)"),
    student=dict(lr=LR_S, weight_decay=WD_S, epochs=EPOCHS_S, patience=PATIENCE_S,
                 hidden_size=HIDDEN, dropout=DROPOUT, channal_attention=CHAN_ATTEN,
                 temporal_attention=TEMP_ATTEN, normalize=False, scheduler=None),
)

## 主流程：下载一个 → 训教师 → 扫 alpha 训学生 → 融合 → 测试

1. `ct.download_patient` 下 EDF
2. `t.preprocess` 切窗打标签
3. `ct.student_dataset` 按 LOOCV 建折索引（丢掉任一侧凑不出 1:1 的折，不建自编码器那份大张量）
4. `t.teacher_from_student` 教师的折向学生看齐，ictal 与间期 1:1
5. `t.fold_norm_stats` 每折一套 mean/std，**只给 teacher 用**（排除两边 val/test 窗）
6. 每折训一次教师（标准化输入），冻结；对每个 alpha 训六个学生（**不标准化**，跟 261105 一致）
7. 每折每 alpha：val 对半分训融合、B 半选阈值；概率平均做基线；测测试集
8. 结果存 Drive，删本地 EDF

In [ ]:
torch.backends.cudnn.benchmark = True   # 固定形状，让 cudnn 选最快的卷积算法


def distill_loss(student_logits, y_student, teacher_logits, alpha, T):
    """FGL：alpha*CE(student, y) + (1-alpha)*T^2*BCE(student/T, teacher/T)
    teacher_logits 是冻结后预先算好的常量，不用每步再前向
    """
    ce = F.binary_cross_entropy_with_logits(student_logits.squeeze(-1), y_student.squeeze(-1))
    with torch.no_grad():
        p_t = torch.sigmoid(teacher_logits / T).squeeze(-1)
    distill = F.binary_cross_entropy_with_logits((student_logits / T).squeeze(-1), p_t) * T ** 2
    return alpha * ce + (1 - alpha) * distill


def make_batches(*ts, shuffle=False, batch_size=32, generator=None):
    """张量已经在 GPU 上，直接按索引切片，省掉 DataLoader 的逐样本 collate 和 H2D 拷贝
    多线程训练时每个模型传自己的 generator，避免抢全局 RNG
    """
    n = len(ts[0])
    if shuffle:
        order = torch.randperm(n, device=ts[0].device, generator=generator)
    else:
        order = torch.arange(n, device=ts[0].device)
    for i in range(0, n, batch_size):
        idx = order[i:i + batch_size]
        yield tuple(x[idx] for x in ts)


def predict(model, x, y, batch_size=BATCH):
    probs, targets = [], []
    model.eval()
    with torch.no_grad():
        for bx, by in make_batches(x, y, batch_size=batch_size):
            probs.append(torch.sigmoid(model(bx)).ravel())
            targets.append(by.ravel())
    return torch.cat(probs).cpu().numpy(), torch.cat(targets).cpu().numpy()


In [ ]:
for pid in PATIENTS:
    done = OUT / pid / ("%s.json" % pid)
    if done.exists():
        print("[跳过] 已完成", pid)
        continue
    print("=" * 15, pid, "=" * 15)
    out = OUT / pid
    weight_dir = out / "weight"
    weight_dir.mkdir(parents=True, exist_ok=True)
    now = datetime.now().strftime("%y%m_%d_%H%M%S")

    # 1. 下载 + 预处理
    pdir = ct.download_patient(pid, DATA)
    windows, meta = t.preprocess(str(pdir), SPH, SOP, INTER_THR, WIN_LEN, WIN_OL, ELECS)

    # 2. 学生折（student_dataset 已丢掉任一侧凑不出 1:1 的折）+ 教师折（对齐）
    student_ds = ct.student_dataset(ELECS, meta, SEED, (ENC_VAL, CLS_VAL))
    teacher_ds = t.teacher_from_student(ELECS, meta, windows, student_ds[0], SEED)
    keep = {d["seizure_id"] for d in teacher_ds}
    student_ds = [[f for f in elec if f["seizure_id"] in keep] for elec in student_ds]
    # 标准化参数只给 teacher 用（统计池排除两边的 val/test 窗）
    norm_stats = [t.fold_norm_stats(meta, windows, ELECS, s, d)
                  for s, d in zip(student_ds[0], teacher_ds)]
    print("[%s] 保留 %d 折 %s" % (pid, len(teacher_ds), [d["seizure_id"] for d in teacher_ds]))

    data = {"config": CONFIG, "teacher": {}, "students": {}, "test_index": []}

    for fold, d in enumerate(teacher_ds):
        mean, std = norm_stats[fold]
        print("---- fold %d/%d seizure %s ----" % (fold + 1, len(teacher_ds), d["seizure_id"]))

        # 3. 教师：六通道 ictal vs 间期，吃标准化后的数据
        t_tr = (t.norm_wins(d["train_wins"], mean, std).to(t.DEVICE), d["y_train"].to(t.DEVICE))
        t_va = (t.norm_wins(d["val_wins"], mean, std).to(t.DEVICE), d["y_val"].to(t.DEVICE))
        torch.manual_seed(SEED * 1000 + fold)      # 教师初始化、打乱可复现
        teacher = Detection().to(t.DEVICE)
        opt_t = torch.optim.Adam(teacher.parameters(), lr=LR_T)
        sched_t = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt_t, mode="min", factor=0.5, patience=5, min_lr=1e-6)
        best_val, best_state, counter = float("inf"), {}, 0
        for epoch in range(EPOCHS_T):
            teacher.train()
            for x, y in make_batches(*t_tr, shuffle=True, batch_size=BATCH):
                opt_t.zero_grad()
                loss = F.binary_cross_entropy_with_logits(teacher(x), y)
                loss.backward()
                opt_t.step()
            teacher.eval()
            val_loss, nb = 0, 0
            with torch.no_grad():
                for x, y in make_batches(*t_va, batch_size=BATCH):
                    val_loss += F.binary_cross_entropy_with_logits(teacher(x), y).item()
                    nb += 1
            val_loss /= nb
            sched_t.step(val_loss)
            if val_loss < best_val:
                best_val, counter = val_loss, 0
                best_state = {k: v.cpu().clone() for k, v in teacher.state_dict().items()}
            else:
                counter += 1
                if counter >= PATIENCE_T:
                    break
        teacher.load_state_dict(best_state)
        teacher.eval()
        teacher.requires_grad_(False)
        t_path = weight_dir / ("%s_teacher_fold%d.pth" % (now, fold + 1))
        torch.save(best_state, t_path)
        data["teacher"]["fold_%d" % (fold + 1)] = {"weight": str(t_path), "val_loss": best_val}
        del t_tr, t_va

        # 4. 教师冻结后，软标签每折只算一遍：6 alpha × 6 电极共用（教师侧仍用标准化后的六通道）
        s0 = student_ds[0][fold]
        x6 = {key: t.norm_wins(torch.tensor(
                  np.stack([windows[en][s0[key + "_idx"]] for en in ELECS], axis=1),
                  dtype=torch.float32), mean, std).to(t.DEVICE)
              for key in ("train", "val", "test")}
        with torch.no_grad():
            t_log = {key: teacher(x6[key]) for key in ("train", "val", "test")}
        del x6
        torch.cuda.empty_cache()

        # 5. 学生输入：不标准化，跟 261105 一致（原始带通信号，直接放 GPU）
        sw = []
        for e, elec in enumerate(ELECS):
            s = student_ds[e][fold]
            dd = {}
            for key in ("train", "val", "test"):
                idx = s[key + "_idx"]
                dd[key + "_wins"] = torch.tensor(windows[elec][idx][:, None, :],
                                                 dtype=torch.float32).to(t.DEVICE)
                dd["y_" + key] = torch.tensor(meta[elec].iloc[idx].label.to_numpy(),
                                              dtype=torch.float32).unsqueeze(1).to(t.DEVICE)
            sw.append(dd)

        def train_one(e, alpha):
            """单个电极的学生：自己一个 Generator，不抢全局 RNG"""
            s = sw[e]
            gen = torch.Generator(device=t.DEVICE)
            gen.manual_seed(SEED * 1000 + fold * 100 + e)   # 只跟 (折, 电极) 有关，六个 alpha 共用
            student = students[e]
            opt_s = torch.optim.Adam(student.parameters(), lr=LR_S, weight_decay=WD_S)
            best_val, best_state, counter = float("inf"), {}, 0
            for epoch in range(EPOCHS_S):
                student.train()
                for x, tl, y in make_batches(s["train_wins"], t_log["train"], s["y_train"],
                                             shuffle=True, batch_size=BATCH, generator=gen):
                    opt_s.zero_grad()
                    loss = distill_loss(student(x), y, tl, alpha, T)
                    loss.backward()
                    opt_s.step()
                student.eval()
                val_loss, nb = 0, 0
                with torch.no_grad():
                    for x, tl, y in make_batches(s["val_wins"], t_log["val"], s["y_val"],
                                                 batch_size=BATCH):
                        val_loss += distill_loss(student(x), y, tl, alpha, T).item()
                        nb += 1
                val_loss /= nb
                if val_loss < best_val:
                    best_val, counter = val_loss, 0
                    best_state = {k: v.cpu().clone() for k, v in student.state_dict().items()}
                else:
                    counter += 1
                    if counter >= PATIENCE_S:
                        break
            student.load_state_dict(best_state)
            vp, vt = predict(student, s["val_wins"], s["y_val"])
            tp, tt = predict(student, s["test_wins"], s["y_test"])
            return e, best_state, vp, tp, vt, tt

        for alpha in ALPHAS:
            # 6 个模型先按固定种子顺序初始化（与串行时一致），再并行训练
            students = []
            for e in range(len(ELECS)):
                torch.manual_seed(SEED * 1000 + fold * 10 + e)
                students.append(ConvBILSTMClassify(channal_attention=CHAN_ATTEN,
                                                   temporal_attention=TEMP_ATTEN,
                                                   dropout=DROPOUT, hidden_size=HIDDEN).to(t.DEVICE))
            with ThreadPoolExecutor(max_workers=len(ELECS)) as ex:
                outs = list(ex.map(lambda e: train_one(e, alpha), range(len(ELECS))))

            val_probs, test_probs, elec_metrics = [], [], []
            vt = tt = None
            for e, best_state, vp, tp, vt_e, tt_e in outs:
                elec = ELECS[e]
                s_path = weight_dir / ("%s_a%.1f_fold%d_%s.pth" % (now, alpha, fold + 1, elec))
                torch.save(best_state, s_path)
                data["students"].setdefault("a%.1f" % alpha, {}).setdefault("fold_%d" % (fold + 1), {})[elec] = str(s_path)
                val_probs.append(vp)
                test_probs.append(tp)
                vt, tt = vt_e, tt_e
                elec_metrics.append(dict(elec=elec, val_auc=float(roc_auc_score(vt_e, vp)),
                                         **ct.classification_metrics(tt_e, tp)))
            del students, outs
            torch.cuda.empty_cache()

            # 6. 电极融合：全 val 拟合，跟 261105 一样，不再选阈值
            fusion = ct.train_fusion(val_probs, vt)
            x_te = torch.tensor(np.array(test_probs).T, dtype=torch.float32).unsqueeze(-1).to(t.DEVICE)
            with torch.no_grad():
                fused = fusion(x_te).cpu().numpy().ravel()
            m_fuse = ct.classification_metrics(tt, fused)

            # 7. 概率平均集成：六路概率直接取平均，可部署的真实基线
            tp_mean = np.mean(test_probs, axis=0)
            m_pmean = dict(val_auc=float(roc_auc_score(vt, np.mean(val_probs, axis=0))),
                           **ct.classification_metrics(tt, tp_mean))

            data["test_index"].append(dict(fold=fold, seizure_id=int(d["seizure_id"]), alpha=alpha,
                                           val_auc_mean=float(np.mean([m["val_auc"] for m in elec_metrics])),
                                           metrics=elec_metrics,
                                           metrics_atten=[m_fuse], metrics_pmean=[m_pmean]))
            ms = {k: np.mean([m[k] for m in elec_metrics])
                  for k in ("auc", "accuracy", "sensitivity", "specificity")}
            print("  a %.1f | 单电极 val %.3f test AUC %.4f ACC %.4f SEN %.4f SPE %.4f"
                  " | 融合 AUC %.4f ACC %.4f SEN %.4f SPE %.4f"
                  " | 平均 AUC %.4f ACC %.4f SEN %.4f SPE %.4f"
                  % (alpha, np.mean([m["val_auc"] for m in elec_metrics]), ms["auc"], ms["accuracy"],
                     ms["sensitivity"], ms["specificity"], m_fuse["auc"], m_fuse["accuracy"],
                     m_fuse["sensitivity"], m_fuse["specificity"],
                     m_pmean["auc"], m_pmean["accuracy"], m_pmean["sensitivity"], m_pmean["specificity"]))
            del fusion, x_te

        del t_log, teacher, sched_t, sw
        torch.cuda.empty_cache()
        gc.collect()

    with open(out / ("%s.json" % pid), "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    del student_ds, teacher_ds, norm_stats, windows, meta, data
    torch.cuda.empty_cache()
    gc.collect()
    ct.clear_patient(pid, DATA)
    print("[完成]", pid)


## 汇总

In [ ]:
df = ct.summarize_distill(OUT)
print(df.groupby("alpha")[["auc", "auc_atten"]].mean())